In [ ]:
# Setup

%load_ext autoreload
%autoreload 2

import json
import logging
from pathlib import Path
from typing import Any

from dotenv import load_dotenv
from pydantic import BaseModel

from comments import prompts
from comments.llm_generator import LLMGenerator
from comments.llm_labeling import LLMLabeler
from comments.static_labeling import (
    label_unfinished_comments,
    label_code_snippets_comments,
)
from shared import (
    DifferencesEvaluator,
    load_dataset,
    save_dataset,
    ManualLabeler,
    add_filename_suffix,
)
from shared.deduplicate import JsonDeduplicator
from shared.llm_connector import Model
from shared.notebooks import comma_separated_paths
from shared.utils import map_labels, dataset_remove_properties, find_by_key
from shared.sampling import random_undersample
from datasets import (
    load_dataset as load_dataset_hf,
    concatenate_datasets,
    load_from_disk,
)

load_dotenv()

logging.basicConfig(level=logging.INFO)
deduplicated_file_path = None

In [ ]:
# Deduplication

input_path = "/media/martin/Big data1/datasets/github-projects/hive.json"
deduplicator = JsonDeduplicator()
file_path = Path(input_path)

deduplicated_file_path = deduplicator.deduplicate_dataset_file(file_path)

In [ ]:
# Labeling unfinished comments - LLM

UNFINISHED_COMMENT_ATTR = "unfinished_comment_llm"

if deduplicated_file_path is None:
    deduplicated_file_path = Path(input("Path to deduplicated file: "))


class ResponseModel(BaseModel):
    is_unfinished: bool


def process_response(response: ResponseModel, element: dict[str, Any]):
    element[UNFINISHED_COMMENT_ATTR] = 1 if response.is_unfinished else 0


labeler = LLMLabeler[ResponseModel](
    init_prompt=prompts.TODO_COMMENTS_INIT_PROMPT,
    run_prompt=prompts.RUN_PROMPT_1,
    labeled_attribute=UNFINISHED_COMMENT_ATTR,
    model=Model.LLAMA_3_1,
    process_response=process_response,
    response_class=ResponseModel,
)
await labeler.label_dataset(deduplicated_file_path, "-llama-labeled-unfinished")

In [ ]:
# Labeling unfinished comments - static rules

if deduplicated_file_path is None:
    deduplicated_file_path = Path(input("Path to deduplicated file: "))

label_unfinished_comments(deduplicated_file_path)

In [ ]:
# Labeling unfinished comments - finding differences

if deduplicated_file_path is None:
    deduplicated_file_path = Path(input("Path to deduplicated file: "))

evaluator = DifferencesEvaluator(
    new_key_name="unfinished_comment_label",
    to_compare_keys=["unfinished_comment_llm", "unfinished_comment_static"],
    no_conflict_key="unfinished_comment_static",
)
dataset = load_dataset(deduplicated_file_path)
evaluator.evaluate(dataset)
save_dataset(deduplicated_file_path, dataset)

In [ ]:
# Labeling commented code comments - LLM

CODE_COMMENT_ATTR = "code_comment_llm"

if deduplicated_file_path is None:
    deduplicated_file_path = Path(input("Path to deduplicated file: "))


class ResponseModel(BaseModel):
    is_code_comment: bool


def process_response(response: ResponseModel, element: dict[str, Any]):
    element[CODE_COMMENT_ATTR] = 1 if response.is_code_comment else 0


labeler = LLMLabeler[ResponseModel](
    init_prompt=prompts.CODE_COMMENTS_INIT_PROMPT_2,
    run_prompt=prompts.RUN_PROMPT_1,
    labeled_attribute=CODE_COMMENT_ATTR,
    model=Model.LLAMA_3_1,
    process_response=process_response,
    response_class=ResponseModel,
)
await labeler.label_dataset(deduplicated_file_path, "-llm-labeled")

In [ ]:
# Labeling commented code comments - static rules

if deduplicated_file_path is None:
    deduplicated_file_path = Path(input("Path to deduplicated file: "))

label_code_snippets_comments(deduplicated_file_path)

In [ ]:
# Labeling commented code comments - finding differences

if deduplicated_file_path is None:
    deduplicated_file_path = Path(input("Path to deduplicated file: "))

evaluator = DifferencesEvaluator(
    new_key_name="code_comment_label",
    to_compare_keys=["code_comment_llm", "code_comment_static"],
    no_conflict_key="code_comment_static",
)
dataset = load_dataset(deduplicated_file_path)
evaluator.evaluate(dataset)
save_dataset(deduplicated_file_path, dataset)

In [ ]:
# Map LLM a static labels to final label

if deduplicated_file_path is None:
    deduplicated_file_path = Path(input("Path to deduplicated file: "))

mapping = {
    "code_comment_static": "code_comment",
    "unfinished_comment_static": "technical_debt",
}

dataset = load_dataset(deduplicated_file_path)
dataset = map_labels(mapping, "clean", dataset)

save_dataset(deduplicated_file_path, dataset)

In [ ]:
# Remove unused dataset properties

if deduplicated_file_path is None:
    deduplicated_file_path = Path(input("Path to deduplicated file: "))

dataset = load_dataset(deduplicated_file_path)
dataset_remove_properties(
    [
        "unfinished_comment_llm",
        "unfinished_comment_static",
        "unfinished_comment_label",
        "code_comment_llm",
        "code_comment_static",
        "code_comment_label",
    ],
    dataset,
)

save_dataset(deduplicated_file_path, dataset)

In [ ]:
# Manual labeling

if deduplicated_file_path is None:
    deduplicated_file_path = Path(input("Path to deduplicated file: "))

# labeler = ManualLabeler(lambda el: el["unfinished_comment_static"] == 1, "unfinished_comment_static")
labeler = ManualLabeler(
    lambda el: el["code_comment_static"] == 1, "code_comment_static"
)
labeler.evaluate_labeled(
    deduplicated_file_path,
)

In [ ]:
# LLM - generate TODO comments

if deduplicated_file_path is None:
    deduplicated_file_path = Path(input("Path to deduplicated file: "))

generator = LLMGenerator(
    prompts.GENERATE_VARIATIONS_TODO_2, deduplicated_file_path, "todo-generated.json"
)


def filter_fnc(element):
    return "technical_debt" in element["labels"]


await generator.for_each_element(filter_fnc)

In [ ]:
# LLM - generate code comments

if deduplicated_file_path is None:
    deduplicated_file_path = Path(input("Path to deduplicated file: "))

generator = LLMGenerator(
    prompts.GENERATE_VARIATIONS_CODE,
    deduplicated_file_path,
    "code-comments-generated.json",
)


def filter_fnc(element):
    return "code_comment" in element["labels"]


await generator.for_each_element(filter_fnc)

In [ ]:
# Merge generated into the dataset

generated_files = comma_separated_paths()
if deduplicated_file_path is None:
    deduplicated_file_path = Path(input("Path to deduplicated file: "))

dataset = load_dataset(deduplicated_file_path)


def process_file(p: Path):
    with open(p, "r") as f:
        content = json.load(f)

    for element in content:
        original_value = find_by_key(dataset, "text", element["origin"])

        for generated_text in element["generated"]:
            new_value = original_value.copy()
            new_value["text"] = generated_text
            dataset.append(new_value)


for file in generated_files:
    path = Path(file)
    process_file(path)

save_dataset(deduplicated_file_path, dataset)

In [ ]:
# Squash, clean and convert to the arrow format


input_paths = comma_separated_paths()
output_path = input("Insert output path: ")

dataset = concatenate_datasets(
    [load_dataset_hf("json", data_files=path)["train"] for path in input_paths]
)
dataset.save_to_disk(output_path)


def clean_data(data):
    text = data["text"]

    if text.startswith("*"):
        data["text"] = text.removeprefix("*").strip()

    if text.startswith("//"):
        data["text"] = text.removeprefix("//").strip()

    return data


cleaned = dataset.map(lambda data: clean_data(data))
cleaned.save_to_disk(output_path)

In [ ]:
# Random undersampling of imbalanced classes

from collections import Counter

path = input("Insert dataset path: ")
dataset = load_from_disk(path, keep_in_memory=False)

undersampled = random_undersample(dataset, "clean", 3500)

all_labels = [label for item in undersampled for label in item["labels"]]
label_counts = Counter(all_labels)
print(label_counts)

new_path = add_filename_suffix(Path(path), "-undersampled")
undersampled.save_to_disk(new_path)

In [ ]:
path = input("Insert dataset path: ")
dataset = load_from_disk(path, keep_in_memory=False)

split = dataset.train_test_split(test_size=0.2)

new_path = add_filename_suffix(Path(path), "-splitted")
split.save_to_disk(new_path)

In [ ]:
# Train

from comments.train import CommentsTrainer

path = input("Insert dataset path: ")
output_path = input("Insert output path: ")
classes = ["clean", "code_comment", "technical_debt"]
special_tokens = ["JAVADOC", "LINE", "BLOCK"]

trainer = CommentsTrainer(classes, special_tokens)

dataset = load_from_disk(path, keep_in_memory=False)
trainer.train_model(dataset, output_path)

In [ ]:
# Evaluate

from comments.train import CommentsTrainer

dataset_path = input("Insert dataset path: ")
model_path = input("Insert model path: ")
dataset = load_from_disk(dataset_path, keep_in_memory=False)


classes = ["clean", "code_comment", "technical_debt"]
special_tokens = ["JAVADOC", "LINE", "BLOCK"]
trainer = CommentsTrainer(classes, special_tokens)

results = trainer.evaluate(model_path, dataset)
print(results)

In [ ]:
# Test - extend dataset with predictions

from transformers import (
    TextClassificationPipeline,
    AutoTokenizer,
    AutoModelForSequenceClassification,
)

model_path = input("Insert model path: ")
dataset_path = input("Insert path to the dataset (arrow format): ")

tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForSequenceClassification.from_pretrained(model_path)
pipeline = TextClassificationPipeline(
    model=model, tokenizer=tokenizer, top_k=None, device=-1
)

dataset = load_from_disk(dataset_path)


def process(elements):
    texts = elements["text"].copy()
    comment_types = elements["commentType"]

    for index, text in enumerate(texts):
        texts[index] = f"[{comment_types[index]}] {text}"

    results = pipeline(texts)
    elements["predictions"] = results

    return elements


mapped = dataset.select(range(1500)).map(
    lambda elements: process(elements), batched=True, batch_size=50
)
output_path = input("Insert output path: ")
mapped.to_json(output_path, lines=True, indent=3)

In [1]:
tokenizer = AutoTokenizer.from_pretrained(
    "/media/martin/Big data1/models/comments-codebert/model"
)
model = AutoModelForSequenceClassification.from_pretrained(
    "/media/martin/Big data1/models/comments-codebert/model"
)
pipeline = TextClassificationPipeline(
    model=model, tokenizer=tokenizer, top_k=None, device=-1
)

while True:
    query = input("Enter text: ")
    if query == "end":
        break
    results = pipeline(query)
    print(results)

NameError: name 'AutoTokenizer' is not defined

In [5]:
from transformers import (
    AutoTokenizer,
)
from optimum.onnxruntime import ORTModelForSequenceClassification

model_path = input("Enter model path: ")
output = input("Enter output path: ")

tokenizer = AutoTokenizer.from_pretrained(model_path)

model = ORTModelForSequenceClassification.from_pretrained(model_path)

model.save_pretrained(output)
tokenizer.save_pretrained(output)

No ONNX files were found for /media/martin/BigData/models/comments-codebert/model/, setting `export=True` to convert the model to ONNX. Don't forget to save the resulting model with `.save_pretrained()`


('/media/martin/BigData/models/comments-codebert/onnx-model/tokenizer_config.json',
 '/media/martin/BigData/models/comments-codebert/onnx-model/special_tokens_map.json',
 '/media/martin/BigData/models/comments-codebert/onnx-model/vocab.json',
 '/media/martin/BigData/models/comments-codebert/onnx-model/merges.txt',
 '/media/martin/BigData/models/comments-codebert/onnx-model/added_tokens.json',
 '/media/martin/BigData/models/comments-codebert/onnx-model/tokenizer.json')